In [ ]:
# model_drink

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

# Load Data
df = pd.read_csv("Dataset/Features.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Create Drink-Specific Target, find the next time 'drink' appears in the activity list
drink_times = df[df['activity'] == 'drink']['timestamp']
df['min_until_drink'] = df['timestamp'].apply(lambda x: min([((t - x).total_seconds()/60) for t in drink_times if t > x], default=999))
df["target_drink"] = (df["min_until_drink"] <= 30).astype(int)

# Train Model
X = df[["hour", "day_of_week", "is_weekend", "minutes_since_midnight", "time_since_last_drink", "time_since_last_eat", "time_since_last_outside"]].fillna(0)
y = df["target_drink"]

model_drink = RandomForestClassifier(n_estimators=100, random_state=42)
model_drink.fit(X, y)

# Save
joblib.dump(model_drink, "model_drink.roboai")
print("Drink Model Saved!")

KeyboardInterrupt: 

In [ ]:
# model_eat

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

# Load Data
df = pd.read_csv("Dataset/Features.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Create Eat-Specific Target
eat_times = df[df['activity'] == 'eat']['timestamp']
df['min_until_eat'] = df['timestamp'].apply(lambda x: min([((t - x).total_seconds()/60) for t in eat_times if t > x], default=999))
df["target_eat"] = (df["min_until_eat"] <= 30).astype(int)

# 3. Train Model
X = df[["hour", "day_of_week", "is_weekend", "minutes_since_midnight", "time_since_last_drink", "time_since_last_eat", "time_since_last_outside"]].fillna(0)
y = df["target_eat"]

model_eat = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model_eat.fit(X, y)

# 4. Save
joblib.dump(model_eat, "model_eat.roboai")
print("Eat Model Saved!")

Eat Model Saved!


In [ ]:
# model_outside

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

# 1. Load Data
df = pd.read_csv("Dataset/Features.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 2. Create Outside-Specific Target
# We track 'outside_out' as the moment you decide to go outside
outside_times = df[df['activity'] == 'outside_out']['timestamp']
df['min_until_outside'] = df['timestamp'].apply(lambda x: min([((t - x).total_seconds()/60) for t in outside_times if t > x], default=999))
df["target_outside"] = (df["min_until_outside"] <= 30).astype(int)

# 3. Train Model
X = df[["hour", "day_of_week", "is_weekend", "minutes_since_midnight", "time_since_last_drink", "time_since_last_eat", "time_since_last_outside"]].fillna(0)
y = df["target_outside"]

model_outside = RandomForestClassifier(n_estimators=100, random_state=42)
model_outside.fit(X, y)

# 4. Save
joblib.dump(model_outside, "model_outside.roboai")
print("Outside Model Saved!")

Outside Model Saved!


In [ ]:
# DRINK FORECAST

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import joblib

# Load Data
try:
    daily = pd.read_csv("Dataset/Daily_Features.csv")
except FileNotFoundError:
    print("Error: Daily_Features.csv not found.")
    exit()

# Add Weekly Memory for Drinking
# window=7: looks back at the last week, min_periods=1: Skips blank days and calculates with remaining data
daily['drink_7day_avg'] = daily['drink'].rolling(window=7, min_periods=1).mean()
daily['eat_7day_avg'] = daily['eat'].rolling(window=7, min_periods=1).mean()

# Train Drink Model
features = ['day_of_week', 'is_weekend', 'drink', 'eat', 'drink_7day_avg', 'eat_7day_avg']
X = daily[features].fillna(0)
y_drink = daily['target_drink_tomorrow']

model_daily_drink = RandomForestRegressor(n_estimators=100, random_state=42)
model_daily_drink.fit(X, y_drink)

# Save
joblib.dump(model_daily_drink, "daily_drink_forecast.roboai")
print("Daily Drink Forecast (Weekly Memory) Saved!")

Daily Drink Forecast (Weekly Memory) Saved!


In [3]:
# EAT FORECAST

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import joblib

# Load Data
try:
    daily = pd.read_csv("Dataset/Daily_Features.csv")
except FileNotFoundError:
    print("Error: Daily_Features.csv not found.")
    exit()

# Add Weekly Memory for Eating
# window=7: looks back at the last week, min_periods=1: Skips blank days and calculates with remaining data
daily['drink_7day_avg'] = daily['drink'].rolling(window=7, min_periods=1).mean()
daily['eat_7day_avg'] = daily['eat'].rolling(window=7, min_periods=1).mean()

# Train Eat Model
features = ['day_of_week', 'is_weekend', 'drink', 'eat', 'drink_7day_avg', 'eat_7day_avg']
X = daily[features].fillna(0)
y_eat = daily['target_eat_tomorrow']

model_daily_eat = RandomForestRegressor(n_estimators=100, random_state=42)
model_daily_eat.fit(X, y_eat)

# Save
joblib.dump(model_daily_eat, "daily_eat_forecast.roboai")
print("Daily Eat Forecast (Weekly Memory) Saved!")


Daily Eat Forecast (Weekly Memory) Saved!
